In [1]:
# Install dependencies

!pip install transformers
!pip install sentencepiece
!pip install torch
!pip install newspaper3k
!pip install lxml_html_clean
!pip install rouge_score

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 82.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 11.7 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13540 sha256=21341027398f78e05f00acc55be5e4fbf72b51c6ab09d133780b8b0544a7ba8a
  Stored in directory: /root/.cache/pip/wheels/a5/91/9f/00d66475960891a64867914273fcaf78df6cb04d905b104a2a
  Created wheel for feedfinder2: filename=feedfinder2-0.0.4-py3-none-any.whl size=3341 sha256=df6448344631632fdb4c9507568c727f2187114bca32e025dbc84bdd0767819a
  Stored in directory: /root/.cache/pip/wheels/9f/9f/fb/364871d7426d3cdd4d293dcf7e53d97f1

In [2]:
# Import libraries

import re
import torch

from newspaper import Article
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
# Load model and tokenizer
# Model: mT5 fine-tuned on XL-Sum (includes Indonesian news articles)

MODEL_NAME = 'csebuetnlp/mT5_multilingual_XLSum'

print(f'Loading model: {MODEL_NAME}')
print('This may take a few minutes on first run (model ~1.2GB)...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

print(f'\nModel loaded. Running on: {device}')

Loading model: csebuetnlp/mT5_multilingual_XLSum
This may take a few minutes on first run (model ~1.2GB)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Model loaded. Running on: cuda


In [4]:
# Input article URL

url = 'https://news.detik.com/berita/d-8510315/wanita-tewas-di-hotel-kebayoran-baru-jaksel-ada-luka-di-kepala'

In [5]:
# Download and parse article

article = Article(url, language='id')
article.download()
article.parse()

title = article.title
text  = article.text

print('TITLE:')
print(title)
print('\nARTICLE (first 1500 chars):')
print(text[:1500])
print(f'\nTotal characters: {len(text)}')
print(f'Total words     : {len(text.split())}')

TITLE:
Wanita Tewas di Hotel Kebayoran Baru Jaksel, Ada Luka di Kepala

ARTICLE (first 1500 chars):
Seorang wanita berinisial L (20) ditemukan tewas dalam sebuah hotel berlokasi di Kebayoran Baru, Jakarta Selatan. Saat ditemukan, ada luka pada bagian kepala jasad wanita tersebut.

"Bukan (pembunuhan), meninggal ya. Penyebab kematiannya masih kita diselidiki ya," terang Kapolsek Kebayoran Baru, AKBP Nugrahadi Kusuma, kepada wartawan saat dikonfirmasi, Jumat (29/5/2026).

"Tapi ada luka, luka di, secara kasat mata ada luka di kepala," sambungnya.

SCROLL TO CONTINUE WITH CONTENT

Nugraha mengatakan belum bisa memastikan asal usul luka di kepala korban. Dia mengatakan masih menunggu hasil pemeriksaan visum.

"Nah, itu belum tahu saya. Karena kan kita juga nunggu hasil dari ini, dari hasil visum," jelas Nugrahadi.

Dia menjelaskan, jasad L ditemukan siang tadi. Namun dia belum merinci tentang kronologi penemuan hingga penyebab pasti wanita tersebut ditemukan tewas.

Total characters: 875
T

In [6]:
# Text cleaning function
# Removes noisy artifacts common in scraped Indonesian news articles

def clean_text(text: str) -> str:

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Remove lines that are likely captions or image labels (very short, all caps)
    lines = text.splitlines()
    lines = [
        line for line in lines
        if not (len(line.strip()) < 60 and line.strip().isupper())
    ]
    text = '\n'.join(lines)

    # Collapse extra whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)

    return text.strip()

In [7]:
# Chunk splitter
#
# mT5-XLSum has a 512 token input limit.
# For long articles we split into overlapping word-based chunks,
# summarize each, then concatenate the partial summaries for
# a final summarization pass (hierarchical summarization).

def split_into_chunks(
    text: str,
    max_words: int = 400,
    overlap_words: int = 50
) -> list:

    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + max_words, len(words))
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)

        if end == len(words):
            break

        start += max_words - overlap_words

    return chunks

In [8]:
# Core summarization function
#
# DISESUAIKAN agar identik dengan T5 untuk perbandingan yang adil (fair comparison):
# - max_new_tokens   : 150  (sama dengan T5)
# - min_new_tokens   : 40   (sama dengan T5)
# - num_beams        : 8    (sama dengan T5)
# - length_penalty   : 1.2  (sama dengan T5)
# - no_repeat_ngram  : 3    (sama dengan T5)
# - repetition_penalty: 1.8 (sama dengan T5)
# - use_cache        : True (sama dengan T5)
# - Tidak ada do_sample / temperature / top_k / top_p (pure beam search, sama dengan T5)

WHITESPACE_HANDLER = lambda text: re.sub(r'\n+', ' . ', text.strip())

def summarize_chunk(
    text: str,
    max_new_tokens: int = 150,
    min_new_tokens: int = 40,
    num_beams: int = 8,
    length_penalty: float = 1.2,
    no_repeat_ngram_size: int = 3,
    repetition_penalty: float = 1.8,
) -> str:

    prepared = WHITESPACE_HANDLER(text)

    inputs = tokenizer(
        prepared,
        return_tensors='pt',
        padding='max_length',
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            num_beams=num_beams,
            length_penalty=length_penalty,
            no_repeat_ngram_size=no_repeat_ngram_size,
            repetition_penalty=repetition_penalty,
            early_stopping=True,
            use_cache=True,
        )

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    return summary

In [9]:
# Hierarchical summarizer
#
# Short articles  (<= 400 words) : single-pass summarization
# Long articles   (>  400 words) : chunk -> summarize each -> merge -> final pass
#
# DISESUAIKAN agar identik dengan T5:
# - Partial chunk : max_new_tokens=100, min_new_tokens=25
# - Final pass    : max_new_tokens=200, min_new_tokens=50

def summarize_article(
    text: str,
    chunk_max_words: int = 400,
    chunk_overlap: int = 50,
    verbose: bool = True
) -> str:

    cleaned = clean_text(text)
    word_count = len(cleaned.split())

    if word_count <= chunk_max_words:
        if verbose:
            print(f'Short article ({word_count} words) — single-pass summarization.')
        return summarize_chunk(cleaned)

    chunks = split_into_chunks(cleaned, chunk_max_words, chunk_overlap)

    if verbose:
        print(f'Long article ({word_count} words) — split into {len(chunks)} chunks.')

    partial_summaries = []
    for i, chunk in enumerate(chunks):
        if verbose:
            print(f'  Summarizing chunk {i + 1}/{len(chunks)}...')
        partial = summarize_chunk(
            chunk,
            max_new_tokens=100,
            min_new_tokens=25
        )
        partial_summaries.append(partial)

    merged = ' '.join(partial_summaries)

    if verbose:
        print('  Running final summarization pass on merged chunks...')

    final_summary = summarize_chunk(
        merged,
        max_new_tokens=200,
        min_new_tokens=50
    )

    return final_summary

In [10]:
# Run summarization

summary = summarize_article(text, verbose=True)

print('\n' + '=' * 60)
print('TITLE:')
print(title)
print('\nSUMMARY:')
print(summary)
print('=' * 60)

Short article (119 words) — single-pass summarization.

TITLE:
Wanita Tewas di Hotel Kebayoran Baru Jaksel, Ada Luka di Kepala

SUMMARY:
Seorang wanita berinisial L (20 tahun) ditemukan tewas dalam sebuah hotel berlokasi di Kebayoran Baru, Jakarta Selatan. Namun penyebab kematiannya masih diselidiki.


In [11]:
# Summary statistics

original_word_count = len(text.split())
summary_word_count  = len(summary.split())
compression_ratio   = (
    (original_word_count - summary_word_count) / original_word_count
) * 100

print('STATISTICS')
print(f'Original words : {original_word_count}')
print(f'Summary words  : {summary_word_count}')
print(f'Compression    : {compression_ratio:.2f}%')

STATISTICS
Original words : 124
Summary words  : 22
Compression    : 82.26%


### Validation: ROUGE Scores

Untuk evaluasi abstractive summarizer, kita tetap menggunakan ROUGE sebagai proxy. Karena tidak ada reference summary buatan manusia, artikel asli digunakan sebagai referensi. Perlu diingat bahwa untuk abstractive summarization, skor ROUGE cenderung lebih rendah dibanding extractive — ini **normal**, karena model generate kata-kata baru, bukan menyalin kalimat.

In [12]:
# Validation: ROUGE Scores

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

scores = scorer.score(target=text, prediction=summary)

print('ROUGE Scores (vs. original article):')
print(f'{"Metric":<10} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 44)
for key, value in scores.items():
    print(
        f'{key:<10} '
        f'{value.precision:>10.4f} '
        f'{value.recall:>10.4f} '
        f'{value.fmeasure:>10.4f}'
    )

ROUGE Scores (vs. original article):
Metric      Precision     Recall         F1
--------------------------------------------
rouge1         0.9545     0.1667     0.2838
rouge2         0.7619     0.1280     0.2192
rougeL         0.9091     0.1587     0.2703


In [13]:
# CELL 13 — Load 5 File JSONL Lokal (IndoSum Test Set)
import json
import os

# Helper function untuk flatten nested list (konsisten dengan T5)
def deep_flatten_list_of_strings(nested_list):
    flat_list = []
    for item in nested_list:
        if isinstance(item, list):
            flat_list.extend(deep_flatten_list_of_strings(item))
        elif isinstance(item, str):
            flat_list.append(item)
        else:
            flat_list.append(str(item))
    return flat_list

# Daftar nama file JSONL Anda
files_to_load = ['test.01.jsonl', 'test.02.jsonl', 'test.03.jsonl', 'test.04.jsonl', 'test.05.jsonl']

print("Membaca 5 file JSONL pengujian lokal untuk mT5...")
test_articles = []
test_summaries = []

for file_name in files_to_load:
    if os.path.exists(file_name):
        print(f"  Memuat {file_name}...")
        with open(file_name, 'r', encoding='utf-8') as f:
            for line in f:
                data = json.loads(line)
                article_words = deep_flatten_list_of_strings(data['paragraphs'])
                article_text = " ".join(article_words)

                summary_words = deep_flatten_list_of_strings(data['summary'])
                summary_text = " ".join(summary_words)

                test_articles.append(article_text)
                test_summaries.append(summary_text)
    else:
        print(f"  ⚠️ File '{file_name}' tidak ditemukan. Pastikan sudah di-upload ke panel kiri.")

print(f"\n✅ Selesai! Total artikel data uji yang siap dimasukkan ke mT5: {len(test_articles)}")

Membaca 5 file JSONL pengujian lokal untuk mT5...
  Memuat test.01.jsonl...
  Memuat test.02.jsonl...
  Memuat test.03.jsonl...
  Memuat test.04.jsonl...
  Memuat test.05.jsonl...

✅ Selesai! Total artikel data uji yang siap dimasukkan ke mT5: 18774


In [14]:
# CELL 14 — Fungsi Evaluasi ROUGE untuk Model mT5 (200 Sampel)
import numpy as np
from rouge_score import rouge_scorer as rs

def evaluate_mt5_on_indosum(articles, summaries, max_samples=200):
    """Evaluasi model mT5 pada data IndoSum lokal dan menghitung rata-rata ROUGE."""
    scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    metrics = {"rouge1": [], "rouge2": [], "rougeL": []}
    total = min(max_samples, len(articles))

    print(f"Mengevaluasi {total} artikel menggunakan mT5...")
    print("Catatan: Proses abstractive generation 200 sampel memakan waktu (sekitar 10-15 menit). Silakan bersabar.")

    for i in range(total):
        if (i + 1) % 20 == 0 or i == 0 or (i + 1) == total:
            print(f"  Progress: {i+1}/{total} artikel selesai diproses...")

        article = articles[i]
        ref_summary = summaries[i]

        if not article.strip():
            continue

        predicted_summary = summarize_article(article, verbose=False)

        if predicted_summary.strip() and ref_summary.strip():
            s = scorer.score(target=ref_summary, prediction=predicted_summary)
            for key in metrics:
                metrics[key].append(s[key].fmeasure)

    results = {key: float(np.mean(vals)) if vals else 0.0 for key, vals in metrics.items()}
    return results

In [15]:
# CELL 15 — Jalankan Evaluasi Akhir mT5
# Random seed di-set agar hasil reprodusibel dan konsisten dengan T5
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MAX_SAMPLES = 200

mt5_rouge_results = evaluate_mt5_on_indosum(test_articles, test_summaries, max_samples=MAX_SAMPLES)

line = "─" * 52
print(f"\n{'='*52}")
print(f"  HASIL EVALUASI ROUGE — mT5 Abstractive (IndoSum)")
print(f"{'='*52}")
print(f"  Model    : csebuetnlp/mT5_multilingual_XLSum")
print(f"  Sampel   : {MAX_SAMPLES} artikel data uji")
print(f"  Seed     : {SEED}")
print(f"{'='*52}\n")

print(f"  {'Metrik':<10} {'F1-Score':>12}")
print(f"  {line}")
for metric_key in ["rouge1", "rouge2", "rougeL"]:
    val = mt5_rouge_results[metric_key]
    bar_len = int(val * 40)
    bar = "█" * bar_len + "░" * (40 - bar_len)
    print(f"  {metric_key.upper():<10} {val:>8.4f}  |{bar}|")
print(f"\n{'='*52}")

Mengevaluasi 200 artikel menggunakan mT5...
Catatan: Proses abstractive generation 200 sampel memakan waktu (sekitar 10-15 menit). Silakan bersabar.
  Progress: 1/200 artikel selesai diproses...
  Progress: 20/200 artikel selesai diproses...
  Progress: 40/200 artikel selesai diproses...
  Progress: 60/200 artikel selesai diproses...
  Progress: 80/200 artikel selesai diproses...
  Progress: 100/200 artikel selesai diproses...
  Progress: 120/200 artikel selesai diproses...
  Progress: 140/200 artikel selesai diproses...
  Progress: 160/200 artikel selesai diproses...
  Progress: 180/200 artikel selesai diproses...
  Progress: 200/200 artikel selesai diproses...

  HASIL EVALUASI ROUGE — mT5 Abstractive (IndoSum)
  Model    : csebuetnlp/mT5_multilingual_XLSum
  Sampel   : 200 artikel data uji
  Seed     : 42

  Metrik         F1-Score
  ────────────────────────────────────────────────────
  ROUGE1       0.4064  |████████████████░░░░░░░░░░░░░░░░░░░░░░░░|
  ROUGE2       0.3175  |████████

In [16]:
# CELL 16 — Contoh Kualitatif Hasil Ringkasan mT5
import random

# Gunakan seed yang sama agar sampel kualitatif reprodusibel
random.seed(42)

total_data = len(test_articles)
indices = random.sample(range(total_data), min(3, total_data))
scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

for rank, idx in enumerate(indices, 1):
    article = test_articles[idx]
    ref_abs = test_summaries[idx]

    predicted = summarize_article(article, verbose=False)
    score_abs = scorer.score(target=ref_abs, prediction=predicted)

    print(f"{'='*60}")
    print(f" CONTOH {rank} (Indeks Data ke-{idx})")
    print(f"{'='*60}")
    print(f" REFERENSI ASLI : {ref_abs[:220]}...")
    print(f"\n PREDIKSI mT5   : {predicted[:220]}...")
    print(f"\n ROUGE-1 F1 : {score_abs['rouge1'].fmeasure:.4f} | ROUGE-L F1 : {score_abs['rougeL'].fmeasure:.4f}")
    print()

 CONTOH 1 (Indeks Data ke-3648)
 REFERENSI ASLI : Google Daydream View versi baru memiliki dasar pengoperasian serupa pendahulunya , bertugas mengubah smartphone jadi unit head-mounted display . Handset diposisikan di depan mata , dan dengan sedikit modifikasi pada sist...

 PREDIKSI mT5   : Google meluncurkan perangkat baru yang bisa dinikmati melalui telepon seluler, mulai dari tablet pintar hingga ponsel pintar. BBC News Indonesia mengumpulkan sejumlah hal yang patut Anda ketahui mengenai perangkat terseb...

 ROUGE-1 F1 : 0.1235 | ROUGE-L F1 : 0.0988

 CONTOH 2 (Indeks Data ke-819)
 REFERENSI ASLI : Floyd Mayweather mengaku duel melawan Conor McGregor adalah pertarungan terakhirnya di ring tinju . Mayweather juga memuji McGregor bahwa dia adalah petarung yang hebat dan sukses memberikan kepada penonton   sebuah hibu...

 PREDIKSI mT5   : Floyd Mayweather mengakui bahwa dirinya telah memilih mitra dansa yang tepat dalam pertarungan terakhirnya melawan Conor McGregor di ring tinju...